# 📌 Jev — TypeSafe AI's "System One" Decision Model

![Topic](https://img.shields.io/badge/Topic-Jev%20%2F%20System%20One-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-October%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — Jev is a model you ask multiple-choice, rating or yes/no questions about some data, and it answers with numbers (how likely each answer is) instead of sentences. Your code can then branch on those numbers directly, quickly and cheaply, with nothing to parse. How trustworthy the numbers are is still being studied: an early independent test found that the raw Choice probabilities hide an "I don't know" share, so they do not mean exactly what they appear to.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | standard library only (`math`, `random`, `dataclasses`) |
| Background | transformers, LLM agents, JSON mode. Calibration, probability over discrete options, and Kahneman's System 1/2 are explained below. |


---
## 1. Overview

**Problem.** Agents and pipelines constantly need small *decisions*: route this ticket, is this output safe, how urgent is this? Today you ask an LLM, which writes prose (or JSON) token by token, slowly and at a price you pay per generated token, and you then parse it. Jev is built to skip that. You send it a block of application *state* plus typed *questions*; it returns typed answers with probabilities in one call (TypeSafe docs, 2026).

**Where it sits.** Not a replacement for the LLM, but a fast "decision layer" next to it: the LLM reasons, writes and plans; Jev gates, routes and scores, and ordinary code acts on the result.

**Why "System One".** Kahneman's *Thinking, Fast and Slow* splits thought into **System 1** (fast, intuitive pattern-matching) and **System 2** (slow, deliberate reasoning). TypeSafe borrows the label: an LLM that "thinks step by step" is System 2; Jev gives a quick gut judgement with no reasoning trace (TypeSafe docs, 2026; OpenRouter docs, 2026).

**Key terms for newcomers**

| Term | Meaning |
|------|---------|
| Probability distribution over choices | A list of numbers, one per option, that sum to 1, e.g. `{billing: 0.7, bug: 0.2, other: 0.1}` |
| Calibration | Whether the numbers mean what they say: of all answers given 80% probability, about 80% should be right |

**Naming caveat.** The brief mentions "Jev (Almeida, 2026)" as a possible primary paper. I found no technical paper or report by Diogo Almeida (CEO and co-founder); TypeSafe has not published architecture details or weights (Wikipedia, 2026). I cite the docs and Wikipedia instead, and treat the Almeida citation as **unconfirmed**.

---
## 2. How It Works

Step by step:

### 2.1 Input state
A string, JSON object or array describing the situation (an email, a log, a network flow). Context is limited to 64k tokens per request, of which at most 32k can be the state plus the longest question (TypeSafe docs, 2026). OpenRouter (2026) lists 32k, which matches that state budget.

### 2.2 Typed questions
You attach one or more questions, each one of three primitives (TypeSafe docs, 2026):

| Primitive | Asks | Returns |
|-----------|------|---------|
| **Choice** | Pick one of up to 255 unordered options | `choice`, `probabilities`, `confidence` |
| **Score** | Place the state on an ordered scale of 2–10 levels | `score` (probability-weighted float), `probabilities`, `confidence` |
| **Noul** | Is this statement true? | `noul`: a single probability 0–1 |

### 2.3 Parallel evaluation
Every question is evaluated in parallel and in isolation against the same state (TypeSafe docs, 2026). Secondary sources describe this as a single non-autoregressive pass with no token-by-token generation; the official docs do not use that term. Vendor-claimed latency is 70–500 ms on TypeSafe's internal workflows.
### 2.4 Distribution per question
Each Choice/Score gets a probability for each option or level.
### 2.5 Derived values
`choice` is the most probable option; `score` is the expected level (probability-weighted average); `confidence` runs from 0 (probability spread evenly) to 1 (all probability on one outcome). For Choice the docs give `confidence = (p_max − 1/n) / (1 − 1/n)`; Score uses a probability-weighted distance from the most likely level; Noul has no confidence field, and the docs suggest `|2p − 1|` (TypeSafe docs, 2026). The docs present this as "a sensible default", not a calibration guarantee. The toy below uses the Choice formula.
### 2.6 Your code decides

Thresholds and `if` statements consume the numbers.

```mermaid
flowchart LR
    A[Application state<br/>text / JSON] --> B[Typed questions<br/>Choice, Score, Noul]
    B --> C[Jev<br/>one parallel pass]
    C --> D[Probability distributions<br/>per question]
    D --> E{Deterministic code}
    E -->|noul unsafe > 0.9| F[Block]
    E -->|choice = billing| G[Billing queue]
    E -->|confidence low| H[Escalate to LLM / human]
```

```
state ──► [Choice? Score? Noul?] ──► Jev ──► {opt: p, ...} per question ──► if/else in your code
```

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Speed and cost vs. LLM text generation.** | No text generation, and questions run in parallel. TypeSafe claims 70–500 ms latency and free output tokens, and says it is 40–200x faster and 40–400x cheaper than frontier LLMs on its own internal tasks. Independently, one intrusion-detection pilot measured Jev as 4.8x faster and 3.8x cheaper than a GPT-5.6 Luna baseline. |
| 🟢 | **Typed output vs. JSON/structured-output mode.** | JSON mode constrains the *syntax* of generated text, so you still get one sampled answer with no principled uncertainty. Jev's output type is native (an option, a float), so there is nothing to parse or retry. |
| 🟢 | **Distribution, not a verbalised guess, vs. LLM text generation.** | An LLM saying "I'm 90% sure" is just more text. Jev returns a probability per option, so you can set thresholds, route low-confidence cases elsewhere and log the numbers for monitoring. |
| 🟢 | **No labelled data or training loop vs. a trained classifier.** |  A classifier needs a dataset and retraining when categories change. With Jev you edit the question or option list in natural language. In an intrusion-detection pilot it gave 15x fewer false alarms than a low-data Random Forest and good novel-attack recall, 2026; one dataset, NSL-KDD). |
| 🟢 | **Multiple judgements per call, isolated from one another vs. a chained LLM.** | Many questions on one state run in parallel, one call, and one question's answer cannot contaminate another (TypeSafe docs, 2026). It is also usable as a cheap guard in front of an LLM agent. |
| 🔴 | **Calibration is not what it looks like (vs. its own marketing).** | The Sys1Cal-v1 benchmark found that Jev's binary Choice answers present P(A) + P(¬A) = 1, hiding a suppressed "I don't know" share P(U). Recovering it raised median soft accuracy from 0.771 to 0.978, so the raw Choice probabilities do not mean what they appear to. TypeSafe says its models are "trained for calibrated decisions" but publishes no calibration test. Single benchmark, True/False questions; not yet replicated.|
| 🔴 | **No explanation vs. LLM text generation.** | There is no reasoning trace or free text (OpenRouter docs, 2026), so you cannot inspect *why* it chose something, and it cannot do multi-step reasoning, tool use or drafting. |
| 🔴 | **Opaque and unverifiable vs. a classifier you train.** | Architecture, weights and the RLCD algorithm are unpublished, and training is on synthetic data only. You cannot fine-tune or audit it, and third-party pages disagree with the official docs on details such as the context window (OpenRouter lists 32k; the official limit is 64k per request, 32k for state plus the longest question). |
| 🔴 | **Questions are independent vs. chained LLM reasoning.** | Because each question is evaluated in isolation, a later question cannot use an earlier answer (TypeSafe docs, 2026). Dependent logic has to live in your code, as multiple calls. |
| 🔴 | **Thin independent evidence vs. mature baselines.** | Most speed and accuracy numbers are vendor claims on internal tasks. Independent work is early: a single-run pentest case study with no significance, one IDS pilot, and a report that a stock Qwen3.5-4B, with no training, matches community Jev-style models built on the same backbone. That was not a comparison with Jev itself, but it suggests the approach may not be unique to Jev. A survey of 2,170 public GitHub projects shows fast adoption, mostly for attribute judgement and scoring, but adoption is not evidence of accuracy.|

---
## 4. Toy Code Example

> **Goal:** Mimic Jev-style output (a 3-option Choice and a Noul) with a fake scorer, then branch on the probabilities in plain Python. **No real API calls.** The scorer is keyword logic; `confidence` uses TypeSafe's documented Choice formula.

In [1]:
# ---------------------------------------------------------------------------
# Jev-style mock: typed questions -> probability distributions -> branching
# ---------------------------------------------------------------------------
import math
from dataclasses import dataclass


def softmax(logits):
    m = max(logits)
    exps = [math.exp(x - m) for x in logits]
    total = sum(exps)
    return [e / total for e in exps]


@dataclass
class ChoiceResult:
    choice: str
    probabilities: dict
    confidence: float  # (p_max - 1/n) / (1 - 1/n), as documented by TypeSafe


@dataclass
class NoulResult:
    noul: float  # P(statement is true), 0..1


def mock_choice(state: str, options: list[str], keywords: dict, fallback: str | None = None) -> ChoiceResult:
    """Fake scorer: each keyword hit adds evidence for its option; the fallback gets a small prior."""
    text = state.lower()
    logits = [
        sum(2.0 for kw in keywords.get(opt, []) if kw in text) + (1.0 if opt == fallback else 0.0)
        for opt in options
    ]
    probs = softmax(logits)
    n = len(options)
    confidence = (max(probs) - 1 / n) / (1 - 1 / n)  # 1 = all mass on one option, 0 = even spread
    best = options[max(range(len(options)), key=probs.__getitem__)]
    return ChoiceResult(best, dict(zip(options, probs)), confidence)


def mock_noul(state: str, risky_words: list[str]) -> NoulResult:
    hits = sum(w in state.lower() for w in risky_words)
    return NoulResult(1 / (1 + math.exp(-(2.5 * hits - 1.5))))  # sigmoid


def decide(state: str):
    # Both questions are evaluated independently against the same state.
    route = mock_choice(
        state,
        ["billing", "bug_report", "other"],
        {"billing": ["invoice", "charge", "refund"], "bug_report": ["crash", "error", "stack trace"]},
        fallback="other",
    )
    unsafe = mock_noul(state, ["password", "ssn", "credit card"])

    # Deterministic branching on the numbers
    if unsafe.noul > 0.9:
        action = "BLOCK: likely sensitive data"
    elif route.confidence < 0.2:
        action = "ESCALATE: Jev is unsure, hand to LLM/human"
    else:
        action = f"ROUTE to {route.choice} queue"

    print(f"state: {state!r}")
    print("  choice :", route.choice, {k: round(v, 3) for k, v in route.probabilities.items()})
    print(f"  confidence={route.confidence:.2f}  noul(unsafe)={unsafe.noul:.2f}")
    print("  ->", action, "\n")


decide("I was charged twice on my last invoice, please issue a refund")
decide("The app shows an error and a crash with a stack trace on login")
decide("Hello, just wondering about your opening hours")
decide("I got an error when paying my invoice")
decide("Refund my invoice, my credit card and password are 1234 and hunter2")

state: 'I was charged twice on my last invoice, please issue a refund'
  choice : billing {'billing': 0.991, 'bug_report': 0.002, 'other': 0.007}
  confidence=0.99  noul(unsafe)=0.18
  -> ROUTE to billing queue 

state: 'The app shows an error and a crash with a stack trace on login'
  choice : bug_report {'billing': 0.002, 'bug_report': 0.991, 'other': 0.007}
  confidence=0.99  noul(unsafe)=0.18
  -> ROUTE to bug_report queue 

state: 'Hello, just wondering about your opening hours'
  choice : other {'billing': 0.212, 'bug_report': 0.212, 'other': 0.576}
  confidence=0.36  noul(unsafe)=0.18
  -> ROUTE to other queue 

state: 'I got an error when paying my invoice'
  choice : billing {'billing': 0.422, 'bug_report': 0.422, 'other': 0.155}
  confidence=0.13  noul(unsafe)=0.18
  -> ESCALATE: Jev is unsure, hand to LLM/human 

state: 'Refund my invoice, my credit card and password are 1234 and hunter2'
  choice : billing {'billing': 0.936, 'bug_report': 0.017, 'other': 0.047}
  confidence

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **Jev answers typed questions (Choice, Score, Noul) about a block of state with probabilities, not prose, so your code can branch on the output directly.**
- **It is called "System One" after Kahneman's fast, intuitive thinking: a quick gut judgement with no reasoning trace, sitting next to the slower "System Two" LLM that plans and writes.**
- **All questions run in parallel and in isolation in one call, which is why TypeSafe claims 70-500 ms latency and free output tokens (vendor claim, mostly unverified independently).**
- **Calibration is the open question: the Sys1Cal-v1 benchmark found hidden "I don't know" probability mass, so raw Choice probabilities are miscalibrated until that share is recovered.**
- **Independent evidence is early and narrow (one IDS pilot, one non-significant pentest case study), and a stock Qwen3.5-4B matched community Jev-style models built on the same backbone, so treat Jev as a promising decision layer to test on your own data, not a proven drop-in.**

</div>


---
## 6. Sources & Verification Notes

**Independent papers**

Porcedda, R. (2026). *Jev thinks "I don't know", but doesn't say it: Introducing Sys1Cal-v1 Dataset for Probability Calibration.* arXiv:2609.35342. https://arxiv.org/abs/2609.35342

Severo, P., Quincozes, S. E. & Dias, A. (2026). *Jev-IDS: System One Models for Network Intrusion Detection.* arXiv:2610.01079. https://arxiv.org/abs/2610.01079

dos Santos Barbosa, J. A. (2026). *Calibrated Decision Models for Autonomous Penetration-Testing Harnesses: JEV and Laya as System One Decision Layers for LLM-Driven Pentest Agents.* arXiv:2609.28940. https://arxiv.org/abs/2609.28940

Li, Y. & Wagle, J. (2026). *LLM-as-Jev: LLMs Are Already Jev-Style Decision Models -- When and How to Fine-Tune Them.* arXiv:2610.02076. https://arxiv.org/abs/2610.02076

Ling, G., Xue, M. & Ye, Z. (2026). *Jev in the Wild: A Data-Driven Analysis of the Jev Model's Functionality, Applications and Ecosystem.* arXiv:2609.30216. https://arxiv.org/abs/2609.30216

**Vendor / official documentation**

TypeSafe AI. (2026). *TypeSafe documentation: System One API (POST https://api.typesafe.ai/v1/systemone), Primitives, Confidence, Models and System One concept pages.* https://docs.typesafe.ai

OpenRouter. (2026). *Jev guide (community integration docs).* https://openrouter.ai/docs/guides/community/jev.md

**Secondary**

Wikipedia. (2026). *Jev (AI model).* https://en.wikipedia.org/wiki/Jev_(AI_model)

MindStudio. (2026). *Jev System One model launch.* https://www.mindstudio.ai/blog/jev-system-one-model-launch

byteiota. (2026). *Jev: TypeSafe AI's System One model.* https://byteiota.com/jev-typesafe-ai-system-one-model/


